## Machine Learning Lab 04
### Haider Ali
### 23-NTU-CS-FL-1254


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline


In [ ]:
df = pd.read_csv("creditcard.csv")
X = df.drop("Class", axis=1)
y = df["Class"]
print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(y.value_counts())
print("\nFraud percentage:", round(y.mean() * 100, 3), "%")

Dataset shape: (284807, 31)

Class distribution:
Class
0    284315
1       492
Name: count, dtype: int64

Fraud percentage: 0.173 %


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('smote', SMOTE(random_state=42)),
                ('knn', KNeighborsClassifier())])

In [ ]:
y_pred_initial = pipeline.predict(X_test)

In [ ]:
initial_accuracy = accuracy_score(y_test, y_pred_initial)
initial_f1 = f1_score(y_test, y_pred_initial, zero_division=0)

print("\nInitial KNN (k=5)")
print("Accuracy:", round(initial_accuracy, 4))
print("F1 Score:", round(initial_f1, 4))


Initial KNN (k=5)
Accuracy: 0.9978
F1 Score: 0.5831


In [ ]:
param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('smote', SMOTE(random_state=42)),
                                       ('knn', KNeighborsClassifier())]),
             n_jobs=-1, param_grid={'knn__n_neighbors': [3, 5, 7, 9, 11]},
             scoring='f1', verbose=1)

In [ ]:
print("\nBest k:", grid_search.best_params_["knn__n_neighbors"])
print("Best cross-validation F1:", round(grid_search.best_score_, 4))

# Evaluate tuned model on the untouched test set
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned, zero_division=0)

print("\nTuned KNN")
print("Accuracy:", round(tuned_accuracy, 4))
print("F1 Score:", round(tuned_f1, 4))


Best k: 3
Best cross-validation F1: 0.6308

Tuned KNN
Accuracy: 0.9985
F1 Score: 0.6561


In [ ]:
comparison = pd.DataFrame({
    "Model": ["KNN (k=5)", "Tuned KNN"],
    "Accuracy": [initial_accuracy, tuned_accuracy],
    "F1 Score": [initial_f1, tuned_f1]
})

print("\nModel Comparison:")
print(comparison.to_string(index=False))


Model Comparison:
    Model  Accuracy  F1 Score
KNN (k=5)  0.997841  0.583051
Tuned KNN  0.998473  0.656126
